In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [82]:
df = pd.read_csv("/content/cleaned_dataset.csv")

In [83]:
df


,Organization,Problem Statement Title,Category,PS Number,Theme,Deadline for Idea Submission,Details,combined_text
0,Ministry of Development of North Eastern Regio...,AI-Based early warning and landslide Risk Moni...,Software,SIH26001,Disaster Management,2026-09-20,Background: The North Eastern Region (NER) fre...,AI-Based early warning and landslide Risk Moni...
1,Ministry of Development of North Eastern Regio...,Al-Based Smart Logistics and Accessibility Int...,Software,SIH26002,Smart Automation,2026-09-20,Background: The North Eastern Region (NER) fac...,Al-Based Smart Logistics and Accessibility Int...
2,Ministry of Development of North Eastern Regio...,AI-Based Cognitive Gaming and Memory Assistanc...,Software,SIH26003,Space Technology,2026-09-20,Background: The North Eastern Region (NER) is ...,AI-Based Cognitive Gaming and Memory Assistanc...
3,Ministry of Development of North Eastern Regio...,Al-Assisted Early Detection System for Osteoar...,Hardware,SIH26004,Space Technology,2026-09-20,Background: Osteoarthritis (OA) is one of the ...,Al-Assisted Early Detection System for Osteoar...
4,Ministry of Development of North Eastern Regio...,Solar-Powered Smart Mini Cold Storage System f...,Hardware,SIH26005,Smart Vehicles,2026-09-20,Background: The North Eastern Region (NER) pro...,Solar-Powered Smart Mini Cold Storage System f...
...,...,...,...,...,...,...,...,...
221,AICTE,Student Innovation,Hardware,SIH26222,Smart Education,2026-09-20,Student Innovation-Submit your ideas to addres...,Student Innovation Smart Education Hardware St...
222,AICTE,Student Innovation,Hardware,SIH26223,Disaster Management,2026-09-20,Student Innovation-Disaster management include...,Student Innovation Disaster Management Hardwar...
223,AICTE,Student Innovation,Hardware,SIH26224,Travel & Tourism,2026-09-20,"Student Innovation-Smart education,a concept t...",Student Innovation Travel & Tourism Hardware S...
224,AICTE,Student Innovation,Hardware,SIH26225,Heritage & Culture,2026-09-20,Student Innovation-Challenge your creative min...,Student Innovation Heritage & Culture Hardware...


In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Organization                  226 non-null    object
 1   Problem Statement Title       226 non-null    object
 2   Category                      226 non-null    object
 3   PS Number                     226 non-null    object
 4   Theme                         226 non-null    object
 5   Deadline for Idea Submission  226 non-null    object
 6   Details                       226 non-null    object
 7   combined_text                 226 non-null    object
dtypes: object(8)
memory usage: 14.3+ KB


In [85]:
df.shape

(226, 8)

In [86]:
# STEP 2 — CREATE WEIGHTED TEXT

df["weighted_text"] = (
    df["Problem Statement Title"].astype(str) + " " +
    df["Problem Statement Title"].astype(str) + " " +
    df["Problem Statement Title"].astype(str) + " " +

    df["Theme"].astype(str) + " " +
    df["Theme"].astype(str) + " " +

    df["Category"].astype(str) + " " +

    df["Details"].astype(str)
)

In [87]:
print(df["weighted_text"].iloc[0])

AI-Based early warning and landslide Risk Monitoring System in NER AI-Based early warning and landslide Risk Monitoring System in NER AI-Based early warning and landslide Risk Monitoring System in NER Disaster Management Disaster Management Software Background: The North Eastern Region (NER) frequently faces landslides, flash floods, road blockages, and slope failures due to heavy rainfall, fragile terrain, and unplanned hill cutting. These incidents often disrupt connectivity, damage infrastructure, delay emergency response, and isolate remote villages for days. Currently, monitoring of vulnerable zones is mostly reactive and dependent on manual reporting. There is limited use of real-time predictive systems for identifying high-risk zones and issuing early warnings to authorities and local communities. With increasing climate vulnerability in the region, there is a need for an AI-enabled real-time monitoring and prediction system that can help authorities take preventive action befor

In [88]:
df["weighted_text"].head()


,weighted_text
0,AI-Based early warning and landslide Risk Moni...
1,Al-Based Smart Logistics and Accessibility Int...
2,AI-Based Cognitive Gaming and Memory Assistanc...
3,Al-Assisted Early Detection System for Osteoar...
4,Solar-Powered Smart Mini Cold Storage System f...


In [89]:
# BUILD IMPROVED TF-IDF MODEL
vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(df["weighted_text"])

print("Improved Dataset TF-IDF Shape:", tfidf_matrix.shape)

Improved Dataset TF-IDF Shape: (226, 7922)


In [96]:
# STEP 4 — USER INPUT AND IMPROVED RECOMMENDATIONS
# Get user skills and interests
user_input = input("Enter your skills and interests: ")

# Convert user input using the SAME TF-IDF vectorizer
user_vector = vectorizer.transform([user_input])

# Calculate similarity with all SIH problems
similarity_scores = cosine_similarity(
    user_vector,
    tfidf_matrix
).flatten()

# Get Top 10 highest matching problem indexes
top_10_indices = similarity_scores.argsort()[-10:][::-1]

print("\nTop 10 problem indexes:")
print(top_10_indices)

print("\nTop 10 similarity scores:")
print(similarity_scores[top_10_indices])

Enter your skills and interests: Python Machine Learning Computer Vision Agriculture

Top 10 problem indexes:
[ 69 125 196 213 138 192 209 100 170 110]

Top 10 similarity scores:
[0.10973451 0.1093006  0.10904834 0.10868614 0.10678426 0.10533712
 0.10501055 0.0977895  0.08194051 0.07535347]


In [97]:
# STEP 8.1 — HYBRID RECOMMENDATION SCORING

def calculate_hybrid_score(user_input, problem, similarity_score):

    # Convert everything to lowercase
    user_text = user_input.lower()

    problem_title = str(problem["Problem Statement Title"]).lower()
    problem_theme = str(problem["Theme"]).lower()
    problem_text = str(problem["weighted_text"]).lower()

    # Base score from TF-IDF similarity
    final_score = similarity_score * 100

    # Skills to check
    skills = [
        "python",
        "machine learning",
        "deep learning",
        "computer vision",
        "artificial intelligence",
        "data science",
        "data analysis",
        "natural language processing",
        "nlp",
        "agriculture",
        "iot",
        "robotics",
        "blockchain",
        "cybersecurity",
        "cloud computing"
    ]

    # Check matching skills
    for skill in skills:

        if skill in user_text:

            # Strong bonus if skill appears in title
            if skill in problem_title:
                final_score += 15

            # Medium bonus if skill appears in theme
            elif skill in problem_theme:
                final_score += 10

            # Smaller bonus if skill appears in details/text
            elif skill in problem_text:
                final_score += 5

    return final_score

In [98]:
# STEP 8.2 — CALCULATE HYBRID SCORES FOR ALL SIH PROBLEMS

hybrid_scores = []

for index in range(len(df)):

    problem = df.iloc[index]

    score = calculate_hybrid_score(
        user_input,
        problem,
        similarity_scores[index]
    )

    hybrid_scores.append(score)

# Convert to NumPy array
hybrid_scores = np.array(hybrid_scores)

# Get new Top 10 based on hybrid score
hybrid_top_10_indices = hybrid_scores.argsort()[-10:][::-1]

print("NEW HYBRID TOP 10 INDEXES:")
print(hybrid_top_10_indices)

print("\nNEW HYBRID TOP 10 SCORES:")
print(hybrid_scores[hybrid_top_10_indices])

NEW HYBRID TOP 10 INDEXES:
[ 69 138 110 125 196 213  16 100 170 108]

NEW HYBRID TOP 10 SCORES:
[25.9734513  25.67842614 22.53534663 20.93006046 20.90483434 20.86861421
 20.15415558 19.77895045 18.19405144 18.1707798 ]


In [99]:
# STEP 8.3 — DISPLAY HYBRID TOP 10 RESULTS

print("=" * 120)
print("HYBRID AI RECOMMENDATION — TOP 10 SIH PROBLEMS")
print("=" * 120)

for rank, index in enumerate(hybrid_top_10_indices, start=1):

    # Get problem data
    problem = df.iloc[index]

    # Original TF-IDF score
    tfidf_score = similarity_scores[index] * 100

    # Final hybrid score
    hybrid_score = hybrid_scores[index]

    # Bonus added by hybrid system
    bonus_added = hybrid_score - tfidf_score

    # Find matching skills
    matched_skills = get_matching_keywords(
        user_input,
        problem["weighted_text"]
    )

    print(f"\n{'=' * 120}")
    print(f"RANK {rank}")
    print(f"{'=' * 120}")

    print("\nProblem Title:")
    print(problem["Problem Statement Title"])

    print("\nPS Number:")
    print(problem["PS Number"])

    print("\nTheme:")
    print(problem["Theme"])

    print("\nCategory:")
    print(problem["Category"])

    print("\nMatching Skills:")
    if matched_skills:
        print(", ".join(matched_skills))
    else:
        print("No exact skill phrase match")

    print(f"\nTF-IDF Text Similarity: {tfidf_score:.2f}")
    print(f"Hybrid Bonus Added: +{bonus_added:.2f}")
    print(f"Final Hybrid Score: {hybrid_score:.2f}")

    print("\n" + "-" * 120)

HYBRID AI RECOMMENDATION — TOP 10 SIH PROBLEMS

RANK 1

Problem Title:
To develop an Artificial Intelligence (AI) / Machine Learning (ML) based system for identification, classification, and prediction of different tropical cyclone patterns using multi-source satellite data.

PS Number:
SIH26070

Theme:
Smart Education

Category:
Software

Matching Skills:
machine learning

TF-IDF Text Similarity: 10.97
Hybrid Bonus Added: +15.00
Final Hybrid Score: 25.97

------------------------------------------------------------------------------------------------------------------------

RANK 2

Problem Title:
Hybrid Quantum Machine Learning Platform for Early Disease Detection

PS Number:
SIH26139

Theme:
MedTech / BioTech / HealthTech

Category:
Software

Matching Skills:
machine learning

TF-IDF Text Similarity: 10.68
Hybrid Bonus Added: +15.00
Final Hybrid Score: 25.68

------------------------------------------------------------------------------------------------------------------------

RAN

In [100]:
# STEP 8.4 — IMPROVED HYBRID SCORING WITH GENERIC TITLE PENALTY

def calculate_hybrid_score(user_input, problem, similarity_score):

    # Convert everything to lowercase
    user_text = user_input.lower()

    problem_title = str(problem["Problem Statement Title"]).lower()
    problem_theme = str(problem["Theme"]).lower()
    problem_text = str(problem["weighted_text"]).lower()

    # Base score from TF-IDF similarity
    final_score = similarity_score * 100

    # Skills to check
    skills = [
        "python",
        "machine learning",
        "deep learning",
        "computer vision",
        "artificial intelligence",
        "data science",
        "data analysis",
        "natural language processing",
        "nlp",
        "agriculture",
        "iot",
        "robotics",
        "blockchain",
        "cybersecurity",
        "cloud computing"
    ]

    # Add bonuses for matching skills
    for skill in skills:

        if skill in user_text:

            if skill in problem_title:
                final_score += 15

            elif skill in problem_theme:
                final_score += 10

            elif skill in problem_text:
                final_score += 5

    # Penalize generic problem titles
    generic_titles = [
        "student innovation",
        "innovation"
    ]

    if problem_title.strip() in generic_titles:
        final_score -= 10

    return final_score

In [101]:
# STEP 8.5 — RECALCULATE HYBRID SCORES AND GET NEW TOP 10

hybrid_scores = []

# Calculate updated hybrid score for every SIH problem
for index in range(len(df)):

    problem = df.iloc[index]

    hybrid_score = calculate_hybrid_score(
        user_input,
        problem,
        similarity_scores[index]
    )

    hybrid_scores.append(hybrid_score)


# Convert list to NumPy array
hybrid_scores = np.array(hybrid_scores)


# Get Top 10 problems based on updated hybrid scores
hybrid_top_10_indices = hybrid_scores.argsort()[-10:][::-1]


# Display the updated Top 10
print("=" * 100)
print("UPDATED HYBRID TOP 10 SIH PROBLEMS")
print("=" * 100)

for rank, index in enumerate(hybrid_top_10_indices, start=1):

    problem = df.iloc[index]

    tfidf_score = similarity_scores[index] * 100
    final_hybrid_score = hybrid_scores[index]
    bonus_added = final_hybrid_score - tfidf_score

    print(f"\nRANK {rank}")
    print("-" * 100)

    print("Problem Title:")
    print(problem["Problem Statement Title"])

    print("\nPS Number:")
    print(problem["PS Number"])

    print("\nTheme:")
    print(problem["Theme"])

    print(f"\nTF-IDF Score: {tfidf_score:.2f}")
    print(f"Hybrid Bonus/Penalty: {bonus_added:+.2f}")
    print(f"Final Hybrid Score: {final_hybrid_score:.2f}")

    print("-" * 100)

UPDATED HYBRID TOP 10 SIH PROBLEMS

RANK 1
----------------------------------------------------------------------------------------------------
Problem Title:
To develop an Artificial Intelligence (AI) / Machine Learning (ML) based system for identification, classification, and prediction of different tropical cyclone patterns using multi-source satellite data.

PS Number:
SIH26070

Theme:
Smart Education

TF-IDF Score: 10.97
Hybrid Bonus/Penalty: +15.00
Final Hybrid Score: 25.97
----------------------------------------------------------------------------------------------------

RANK 2
----------------------------------------------------------------------------------------------------
Problem Title:
Hybrid Quantum Machine Learning Platform for Early Disease Detection

PS Number:
SIH26139

Theme:
MedTech / BioTech / HealthTech

TF-IDF Score: 10.68
Hybrid Bonus/Penalty: +15.00
Final Hybrid Score: 25.68
-------------------------------------------------------------------------------------

In [102]:
# STEP 9 — MULTI-SKILL MATCHING

SKILLS = [
    "python",
    "machine learning",
    "deep learning",
    "computer vision",
    "artificial intelligence",
    "data science",
    "data analysis",
    "natural language processing",
    "nlp",
    "agriculture",
    "iot",
    "robotics",
    "blockchain",
    "cybersecurity",
    "cloud computing"
]


def get_user_skills(user_input):

    user_text = user_input.lower()

    detected_skills = []

    for skill in SKILLS:
        if skill in user_text:
            detected_skills.append(skill)

    return detected_skills


def get_matched_skills(user_input, problem_text):

    user_skills = get_user_skills(user_input)

    problem_text = str(problem_text).lower()

    matched_skills = []

    for skill in user_skills:
        if skill in problem_text:
            matched_skills.append(skill)

    return matched_skills


print("Detected User Skills:")
print(get_user_skills(user_input))

Detected User Skills:
['python', 'machine learning', 'computer vision', 'agriculture']


In [103]:
# STEP 10 — DOMAIN AND THEME MATCHING

def get_theme_match(user_input, problem):

    user_text = user_input.lower()

    theme = str(problem["Theme"]).lower()

    matched_domains = []

    domains = [
        "agriculture",
        "health",
        "education",
        "robotics",
        "cybersecurity",
        "space",
        "environment",
        "smart automation",
        "transportation"
    ]

    for domain in domains:

        if domain in user_text and domain in theme:
            matched_domains.append(domain)

    return matched_domains

In [105]:
# STEP 11 — ADVANCED HYBRID RECOMMENDATION SCORE

def calculate_final_score(user_input, problem, similarity_score):

    user_text = user_input.lower()

    problem_title = str(
        problem["Problem Statement Title"]
    ).lower()

    problem_theme = str(
        problem["Theme"]
    ).lower()

    problem_text = str(
        problem["weighted_text"]
    ).lower()

    # Base TF-IDF score
    final_score = similarity_score * 100

    # Get user skills
    user_skills = get_user_skills(user_input)

    # Count matched skills
    matched_skills = get_matched_skills(
        user_input,
        problem_text
    )

    # Bonus for each matched skill
    for skill in matched_skills:

        if skill in problem_title:
            final_score += 12

        elif skill in problem_theme:
            final_score += 8

        else:
            final_score += 4

    # Skill coverage bonus
    if len(user_skills) > 0:

        skill_coverage = (
            len(matched_skills) / len(user_skills)
        )

        final_score += skill_coverage * 20

    # Theme/domain bonus
    theme_matches = get_theme_match(
        user_input,
        problem
    )

    final_score += len(theme_matches) * 10

    # Penalize generic titles
    generic_titles = [
        "student innovation",
        "innovation"
    ]

    if problem_title.strip() in generic_titles:
        final_score -= 10

    return final_score

In [106]:
# STEP 12 — TEST MULTIPLE USER QUERIES

test_queries = [
    "Python Machine Learning Computer Vision Agriculture",
    "Machine Learning Deep Learning Healthcare",
    "Computer Vision Robotics",
    "Python Data Science Data Analysis",
    "Artificial Intelligence Agriculture"
]


for query in test_queries:

    print("\n" + "=" * 100)
    print("USER QUERY:", query)
    print("=" * 100)

    # Convert query into TF-IDF vector
    test_vector = vectorizer.transform([query])

    # Calculate similarity
    test_similarity = cosine_similarity(
        test_vector,
        tfidf_matrix
    ).flatten()

    # Calculate final scores
    test_final_scores = []

    for index in range(len(df)):

        score = calculate_final_score(
            query,
            df.iloc[index],
            test_similarity[index]
        )

        test_final_scores.append(score)

    test_final_scores = np.array(
        test_final_scores
    )

    # Top 3
    top_3_indices = test_final_scores.argsort()[-3:][::-1]

    for rank, index in enumerate(
        top_3_indices,
        start=1
    ):

        print(f"\nRank {rank}")

        print(
            df.iloc[index][
                "Problem Statement Title"
            ]
        )

        print(
            f"Final Score: "
            f"{test_final_scores[index]:.2f}"
        )


USER QUERY: Python Machine Learning Computer Vision Agriculture

Rank 1
Smart Al-Enabled Rapid Feed and Silage Quality Testing System for Dairy Farmers
Final Score: 39.54

Rank 2
Predictive Analytics System for Early Detection of Land Acquisition Delays
Final Score: 37.15

Rank 3
Al-Based Predictive Modelling for Early Forecasting of Bovine Mastitis in lndian Dairy Farms
Final Score: 35.17

USER QUERY: Machine Learning Deep Learning Healthcare

Rank 1
Hybrid Quantum Machine Learning Platform for Early Disease Detection
Final Score: 51.34

Rank 2
OceanEmbed - Satellite Embedding-Based Deep Learning Framework for Reconstruction of Subsurface Ocean Temperature from Surface Satellite Observations.
Final Score: 47.86

Rank 3
To develop an Artificial Intelligence (AI) / Machine Learning (ML) based system for identification, classification, and prediction of different tropical cyclone patterns using multi-source satellite data.
Final Score: 40.04

USER QUERY: Computer Vision Robotics

Rank 1

In [107]:
# STEP 13 — RECOMMENDATION QUALITY EVALUATION

def evaluate_recommendation(query):

    test_vector = vectorizer.transform([query])

    test_similarity = cosine_similarity(
        test_vector,
        tfidf_matrix
    ).flatten()

    final_scores = []

    for index in range(len(df)):

        score = calculate_final_score(
            query,
            df.iloc[index],
            test_similarity[index]
        )

        final_scores.append(score)

    final_scores = np.array(final_scores)

    top_indices = final_scores.argsort()[-10:][::-1]

    evaluation_results = []

    for rank, index in enumerate(
        top_indices,
        start=1
    ):

        problem = df.iloc[index]

        matched_skills = get_matched_skills(
            query,
            problem["weighted_text"]
        )

        evaluation_results.append({
            "Rank": rank,
            "PS Number": problem["PS Number"],
            "Title": problem[
                "Problem Statement Title"
            ],
            "Matched Skills": ", ".join(
                matched_skills
            ),
            "Number of Matches": len(
                matched_skills
            ),
            "Final Score": round(
                final_scores[index],
                2
            )
        })

    return pd.DataFrame(
        evaluation_results
    )


evaluation_df = evaluate_recommendation(
    user_input
)

display(evaluation_df)

,Rank,PS Number,Title,Matched Skills,Number of Matches,Final Score
0,1,SIH26111,Smart Al-Enabled Rapid Feed and Silage Quality...,"computer vision, agriculture",2,39.54
1,2,SIH26017,Predictive Analytics System for Early Detectio...,"machine learning, agriculture",2,37.15
2,3,SIH26109,Al-Based Predictive Modelling for Early Foreca...,"machine learning, agriculture",2,35.17
3,4,SIH26126,Vision Based Autonomous Navigation for Unmanne...,"computer vision, agriculture",2,28.93
4,5,SIH26070,To develop an Artificial Intelligence (AI) / M...,machine learning,1,27.97
5,6,SIH26101,Develop an AI enabled learning platform that i...,"python, machine learning",2,27.78
6,7,SIH26131,Early detection and management of crop disease...,agriculture,1,27.73
7,8,SIH26139,Hybrid Quantum Machine Learning Platform for E...,machine learning,1,27.68
8,9,SIH26132,Strengthening market linkages and price discov...,agriculture,1,26.27
9,10,SIH26171,On-device Visual Perception for Light-weight B...,"machine learning, computer vision",2,26.19


In [108]:
# STEP 14 — SAVE FINAL RECOMMENDATION MODEL

import pickle

# Save TF-IDF vectorizer
with open(
    "sih_tfidf_vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(
        vectorizer,
        file
    )


# Save TF-IDF matrix
with open(
    "sih_tfidf_matrix.pkl",
    "wb"
) as file:

    pickle.dump(
        tfidf_matrix,
        file
    )


# Save processed dataset
df.to_csv(
    "sih_recommendation_dataset.csv",
    index=False
)


print("MODEL SAVED SUCCESSFULLY")

print("\nSaved Files:")
print("- sih_tfidf_vectorizer.pkl")
print("- sih_tfidf_matrix.pkl")
print("- sih_recommendation_dataset.csv")

MODEL SAVED SUCCESSFULLY

Saved Files:
- sih_tfidf_vectorizer.pkl
- sih_tfidf_matrix.pkl
- sih_recommendation_dataset.csv
